In [17]:
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer, InputExample
from sentence_transformers.sentence_transformer import losses
from sentence_transformers.cross_encoder import CrossEncoder
from sentence_transformers.sentence_transformer.datasets import NoDuplicatesDataLoader
from sentence_transformers.sentence_transformer.evaluation import EmbeddingSimilarityEvaluator
from sentence_transformers.sentence_transformer.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.trainer import SentenceTransformerTrainer

# 加载训练数据
train_dataset = load_dataset(
    "nyu-mll/glue", "mnli", split="train"
).select(range(50_000))

mapping = {2: 0, 1: 0, 0: 1}

gold_dataset = train_dataset.select(range(10_000))
gold_examples = [
    InputExample(texts=[row["premise"], row["hypothesis"]], label=mapping[row["label"]])
    for row in gold_dataset]
gold_dataloader = NoDuplicatesDataLoader(gold_examples, batch_size=32)

# 暂存 gold 数据集，待与 silver 数据集合并
gold = pd.DataFrame(
    {
        "sentence1": gold_dataset["premise"],
        "sentence2": gold_dataset["hypothesis"],
        "label": [mapping[label] for label in gold_dataset["label"]]
    }
)

# 在黄金数据集中训练交叉编码器, num_labels=2 很重要，不然 output 全是 1
cross_encoder = CrossEncoder('bert-base-uncased', num_labels=2, device="cuda")
cross_encoder.fit(
    train_dataloader=gold_dataloader,
    epochs=1,
    show_progress_bar=True,
    warmup_steps=100,
    use_amp=False
)

# 取 40,000 条数据组成未标注的数据集
silver_dataset = train_dataset.select(range(10_000, 50_000))
paris = list(zip(silver_dataset["premise"], silver_dataset["hypothesis"]))

# 使用经过微调的交叉编码器标注句子对
output = cross_encoder.predict(paris, apply_softmax=True, show_progress_bar=True)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


Batches:   0%|          | 0/1250 [00:00<?, ?it/s]

In [18]:
output

array([[0.28694683, 0.7130531 ],
       [0.923242  , 0.07675801],
       [0.9736854 , 0.02631468],
       ...,
       [0.8825118 , 0.11748824],
       [0.32596242, 0.6740376 ],
       [0.18731482, 0.8126852 ]], shape=(40000, 2), dtype=float32)

In [19]:

# 获得 silver 数据集，待与 gold 数据集合并
silver = pd.DataFrame(
    {
        "sentence1": silver_dataset["premise"],
        "sentence2": silver_dataset["hypothesis"],
        "label": np.argmax(output, axis=1)
    }
)

# 最终的已标注的数据
data = pd.concat([gold, silver], ignore_index=True, axis=0)
data = data.drop_duplicates(subset=["sentence1", "sentence2"], keep="first")
train_dataset = Dataset.from_pandas(data, preserve_index=False)

# 后面的训练过程就是一样的了 ------------

embedding_model = SentenceTransformer("bert-base-uncased")

# 定义损失函数
train_loss = losses.CosineSimilarityLoss(model=embedding_model)

# 定义评估器，使用语义文本相似度基准(Semantic Textual Similarity Benchmark, STSB)
# 这是一个由人工标注的句子对数据集，相似度分数在 1 ~ 5 之间
val_sts = load_dataset("nyu-mll/glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]], # 值转换为 0~1 之间
    main_similarity="cosine"
)

# 定义训练参数
args = SentenceTransformerTrainingArguments(
    output_dir="augmented_embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True,
    eval_steps=100,
    logging_steps=100
)

# 训练模型
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    loss=train_loss,
    evaluator=evaluator
)

trainer.train()
print(evaluator(embedding_model))
embedding_model.save("augmented_embedding_model")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,0.217919
200,0.158428
300,0.142127
400,0.139490
500,0.139120
600,0.135831
700,0.133818
800,0.131488
900,0.130345
1000,0.129725


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'pearson_cosine': 0.7093859109290936, 'spearman_cosine': 0.7151919508569721}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
evaluator(embedding_model)

{'pearson_cosine': 0.7093859109290936, 'spearman_cosine': 0.7151919508569721}